<a href="https://colab.research.google.com/github/meechkb/retail-sales-forecasting-pipeline/blob/main/Retail_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retail Sales Forecasting Pipeline

## Objective
The goal of this project is to forecast daily retail sales using historical transaction data. This notebook demonstrates an end-to-end pipeline that includes data ingestion, cleaning, feature engineering, time-series transformation, model training, evaluation, and visualization.

In [ ]:
# Import required libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, Input,
    LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
)

sns.set(style="whitegrid")

## Data Ingestion

The dataset is stored as a multi-sheet Excel file. This section loads the two retail transaction sheets and combines them into one dataframe.

In [ ]:
# Define file path

BASE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else "."
file_path = os.path.join(BASE_DIR, "data", "raw", "211FinalProject.xlsx")

if not os.path.exists(file_path):
    raise FileNotFoundError("Dataset not found. Please place it in data/raw/211FinalProject.xlsx")

# Load each sheet
df_2009 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_2010 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

# Combine sheets
df = pd.concat([df_2009, df_2010], ignore_index=True)

df.head()

FileNotFoundError: Dataset not found. Please place it in data/raw/211FinalProject.xlsx

## Data Cleaning and Preprocessing

This section removes missing values, canceled invoices, and invalid records. It also creates the Sales column and prepares the dataset for daily time-series aggregation.

In [ ]:
# Keep only necessary columns
required_columns = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID"]

df = df[required_columns].copy()

# Remove rows with missing values
df.dropna(subset=required_columns, inplace=True)

# Remove canceled invoices
df = df[~df["Invoice"].astype(str).str.startswith("C")].copy()

# Keep only positive quantity and price records
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()

# Create Sales column
df["Sales"] = df["Quantity"] * df["Price"]

# Convert InvoiceDate to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Sort by date
df.sort_values("InvoiceDate", inplace=True)

df.head()

## Daily Sales Aggregation

The transaction-level dataset is aggregated into daily sales totals. Missing dates are filled with 0 to maintain a continuous time-series structure.

In [ ]:
# Aggregate sales by day
daily_sales = df.groupby(df["InvoiceDate"].dt.date)["Sales"].sum().reset_index()

daily_sales.columns = ["Date", "Sales"]
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])
daily_sales.set_index("Date", inplace=True)

# Create continuous daily frequency
daily_sales = daily_sales.asfreq("D")

# Fill missing sales days with 0
daily_sales["Sales"] = daily_sales["Sales"].fillna(0)

daily_sales.head()

## Feature Engineering

This section creates time-based features, scales the sales column, and adds lag and rolling-window features to help the models learn time-series patterns.

In [ ]:
# Time-based features
daily_sales["dayofweek"] = daily_sales.index.dayofweek
daily_sales["month"] = daily_sales.index.month
daily_sales["quarter"] = daily_sales.index.quarter
daily_sales["year"] = daily_sales.index.year
daily_sales["dayofmonth"] = daily_sales.index.day

# Scale Sales column
scaler = MinMaxScaler()
daily_sales["Sales_scaled"] = scaler.fit_transform(daily_sales[["Sales"]])

# Lag and rolling features
daily_sales["lag_1"] = daily_sales["Sales_scaled"].shift(1)
daily_sales["lag_7"] = daily_sales["Sales_scaled"].shift(7)
daily_sales["lag_14"] = daily_sales["Sales_scaled"].shift(14)
daily_sales["rolling_mean_7"] = daily_sales["Sales_scaled"].rolling(window=7).mean()
daily_sales["rolling_std_7"] = daily_sales["Sales_scaled"].rolling(window=7).std()

# Drop rows created with NaN values from lag/rolling features
daily_sales.dropna(inplace=True)

daily_sales.head()

## Time-Series Transformation

The dataset is converted into supervised learning sequences using a 30-day sliding window. Each sequence is used to predict the next day’s scaled sales value.

In [ ]:
# Define features and target
features = ["Sales_scaled", "lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7"]

X_all = daily_sales[features].values
y_all = daily_sales["Sales_scaled"].values

# Create sequences
window_size = 30
X, y = [], []

for i in range(len(X_all) - window_size):
    X.append(X_all[i:i + window_size])
    y.append(y_all[i + window_size])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

## Train-Test Split

The data is split chronologically, using the first 80% for training and the remaining 20% for testing. This preserves the time order of the dataset.

In [ ]:
# Chronological train-test split
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## Model Evaluation Function

This function evaluates each model using MAE, RMSE, and R² score.

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{model_name} Results:")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}\n")

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

## Machine Learning Models

This section trains tree-based regression models using flattened 30-day sequences.

In [ ]:
results = []

# Flatten sequence data for traditional ML models
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train_flat, y_train)
dt_preds = dt_model.predict(X_test_flat)
results.append(evaluate_model(y_test, dt_preds, "Decision Tree"))

# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_flat, y_train)
rf_preds = rf_model.predict(X_test_flat)
results.append(evaluate_model(y_test, rf_preds, "Random Forest"))

# XGBoost
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_flat, y_train)
xgb_preds = xgb_model.predict(X_test_flat)
results.append(evaluate_model(y_test, xgb_preds, "XGBoost"))

## Deep Learning Models

This section trains LSTM, GRU, and Transformer-based models. These models are designed to learn patterns from sequential time-series data.

In [ ]:
# LSTM Model
lstm_model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(units=50, return_sequences=False),
    Dropout(0.2),
    Dense(units=1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

lstm_preds = lstm_model.predict(X_test)
results.append(evaluate_model(y_test, lstm_preds, "LSTM"))

In [ ]:
# GRU Model
gru_model = Sequential([
    GRU(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    GRU(units=50, return_sequences=False),
    Dropout(0.2),
    Dense(units=1)
])

gru_model.compile(optimizer="adam", loss="mean_squared_error")
gru_model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

gru_preds = gru_model.predict(X_test)
results.append(evaluate_model(y_test, gru_preds, "GRU"))

In [ ]:
# Transformer Encoder Function
def transformer_encoder(inputs, head_size, num_heads, ff_dim):
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    attention = LayerNormalization(epsilon=1e-6)(attention)

    ff = Dense(ff_dim, activation="relu")(attention)
    ff = Dense(inputs.shape[-1])(ff)
    ff = LayerNormalization(epsilon=1e-6)(ff)

    return ff

# Transformer Model
input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))
x = transformer_encoder(input_layer, head_size=64, num_heads=4, ff_dim=128)
x = GlobalAveragePooling1D()(x)
output_layer = Dense(1)(x)

transformer_model = Model(inputs=input_layer, outputs=output_layer)
transformer_model.compile(optimizer="adam", loss="mean_squared_error")

transformer_model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

transformer_preds = transformer_model.predict(X_test)
results.append(evaluate_model(y_test, transformer_preds, "Transformer"))

## Results Summary Table

The results are organized into a dataframe to compare model performance.

In [ ]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="RMSE")

## Visualizations

This section creates visual outputs to analyze sales trends, seasonal patterns, feature importance, and model predictions.

In [ ]:
# Create output folder for visuals
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Daily sales trend
# Create output folder if it doesn't exist
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

# Plot daily sales
plt.figure(figsize=(14, 6))
plt.plot(daily_sales.index, daily_sales["Sales"])
plt.title("Daily Sales Trend (2009–2011 Retail Data)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()

# SAVE IMAGE
plt.savefig(os.path.join(output_dir, "daily_sales_trend.png"))

plt.show()

NameError: name 'daily_sales' is not defined

<Figure size 1400x600 with 0 Axes>

In [ ]:
# Sales by day of week
plt.figure(figsize=(10, 5))
sns.boxplot(x="dayofweek", y="Sales", data=daily_sales.reset_index())
plt.title("Sales Distribution by Day of Week")
plt.xlabel("Day of Week (0 = Monday)")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sales_by_dayofweek.png"))
plt.show()

In [ ]:
# Sales by month
plt.figure(figsize=(10, 5))
sns.boxplot(x="month", y="Sales", data=daily_sales.reset_index())
plt.title("Sales Distribution by Month")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sales_by_month.png"))
plt.show()

In [ ]:
# Random Forest feature importance
importances = rf_model.feature_importances_

n_time_steps = X_train.shape[1]
n_features = X_train.shape[2]

reshaped_importances = importances.reshape(n_time_steps, n_features)
avg_importances = reshaped_importances.mean(axis=0)

plt.figure(figsize=(8, 4))
sns.barplot(x=avg_importances, y=features)
plt.title("Random Forest Feature Importance")
plt.xlabel("Average Importance")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "random_forest_feature_importance.png"))
plt.show()

In [ ]:
# XGBoost feature importance
importances_xgb = xgb_model.feature_importances_

reshaped_xgb_importances = importances_xgb.reshape(n_time_steps, n_features)
avg_xgb_importances = reshaped_xgb_importances.mean(axis=0)

plt.figure(figsize=(8, 4))
sns.barplot(x=avg_xgb_importances, y=features)
plt.title("XGBoost Feature Importance")
plt.xlabel("Average Importance")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "xgboost_feature_importance.png"))
plt.show()

In [ ]:
# Actual vs predicted sales using GRU
plt.figure(figsize=(14, 6))
plt.plot(y_test, label="Actual")
plt.plot(gru_preds.flatten(), label="GRU Predicted")
plt.title("GRU Model: Actual vs Predicted Sales")
plt.xlabel("Time Steps")
plt.ylabel("Scaled Sales")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "gru_actual_vs_predicted.png"))
plt.show()

## Final Project Summary

This project demonstrates an end-to-end forecasting pipeline that transforms raw retail transaction data into a structured time-series dataset. The pipeline includes data cleaning, feature engineering, sequence generation, model training, evaluation, and visualization.

The project compares traditional machine learning models with deep learning models to evaluate which approaches perform best for retail sales forecasting. This workflow reflects practical data engineering and machine learning skills, including preprocessing, pipeline design, model comparison, and business-focused analysis.

## Final Results Interpretation

Among the models tested, tree-based models such as Random Forest and gradient boosting methods performed consistently well due to their ability to capture non-linear relationships in structured features.

Deep learning models such as GRU and LSTM demonstrated the ability to capture sequential patterns, though performance depends on hyperparameter tuning and data scaling.

Overall, the results highlight the importance of feature engineering in time-series forecasting and show that both traditional and deep learning models can be effective depending on the use case.